### import

In [6]:
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from neomodel import db, config as neomodel_config
from local import NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD

### connect neo4j

In [7]:
neomodel_config.DATABASE_URL = f'bolt://{NEO4J_USER}:{DB_PASSWORD}@{NEO4J_HOST}:{NEO4J_PORT}'

try:
    result, _ = db.cypher_query("RETURN 1 AS test")
    print("✅ Neo4j 연결 성공! → 응답값:", result[0][0])
except ServiceUnavailable as e:
    print("❌ Neo4j 연결 실패:", e)

✅ Neo4j 연결 성공! → 응답값: 1


### helper function

In [9]:
def cypher_df(q: str) -> pd.DataFrame:
    rec, cols = db.cypher_query(q)
    return pd.DataFrame(rec, columns=cols)

### load data

In [10]:
# load data from neo4j where eval_set = test & role_test = train(target)
gt_df = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order {eval_set:'test', role_test:'train'})
WITH m, o
ORDER BY o.order_number DESC
WITH m, collect(o)[0] AS last_o                
MATCH (last_o)-[:CONTAINS]->(p:Product)
RETURN m.member_id               AS member_id,
       last_o.order_id           AS last_order_id,
       last_o.order_number       AS last_order_number,
       collect(DISTINCT p.product_id) AS products
ORDER BY member_id
""")

gt_df["member_id"]     = gt_df["member_id"].astype(str) #implicit casting!
gt_df["last_order_id"] = gt_df["last_order_id"].astype(str)
gt_df["products"]      = gt_df["products"].apply(
    lambda lst: [str(pid) for pid in lst]      #implicit casting!
)

print(gt_df.head())

  member_id last_order_id  last_order_number  \
0       100       2875733                  5   
1    100002        810045                 12   
2    100005       3275116                 18   
3    100006       1373732                 13   
4     10001       2230385                 22   

                                            products  
0  [48628, 26369, 38689, 27344, 21616, 38547, 248...  
1  [8584, 22750, 44319, 5994, 36086, 27323, 6201,...  
2                                     [49543, 42413]  
3  [22255, 22451, 43504, 21137, 33527, 29926, 135...  
4  [5241, 20947, 25010, 9421, 24852, 45417, 17333...  


### validate data

In [11]:
# ── GT 데이터의 기초 검증 · 통계 ------------------------------------------

# 1) 총 회원 수 (= 행 수)
n_rows = len(gt_df)
print(f"총 회원 수 (행)      : {n_rows:,}")

# 2) member_id 중복 검사
dup_cnt = gt_df["member_id"].duplicated().sum()
print(f"중복 member_id 수     : {dup_cnt}")
assert dup_cnt == 0, "❌ member_id 중복이 존재합니다!"

# 3) products 리스트가 비어 있는 행 수
empty_prod_cnt = gt_df["products"].apply(len).eq(0).sum()
print(f"빈(products=[]) 행 수 : {empty_prod_cnt}")
assert empty_prod_cnt == 0, "❌ 일부 회원의 마지막 주문에 product가 없습니다."

# 4) products 개수 통계
prod_len = gt_df["products"].apply(len)
print("상품 개수 5-수 요약  :", prod_len.describe(percentiles=[.25,.5,.75]).astype(int).to_dict())

# 5) last_order_number 분포 (min / max)
print("last_order_number min :", gt_df["last_order_number"].min())
print("last_order_number max :", gt_df["last_order_number"].max())

# 6) 타입 검증 (optional)
assert gt_df["member_id"].dtype == object,          "member_id dtype 오류"
assert gt_df["last_order_id"].dtype == object,      "last_order_id dtype 오류"
assert gt_df["last_order_number"].dtype.kind in "iu", "last_order_number dtype 오류"  # int/uint

print("\n✅ basic validation complete!")


총 회원 수 (행)      : 75,000
중복 member_id 수     : 0
빈(products=[]) 행 수 : 0
상품 개수 5-수 요약  : {'count': 75000, 'mean': 10, 'std': 7, 'min': 1, '25%': 5, '50%': 9, '75%': 14, 'max': 73}
last_order_number min : 3
last_order_number max : 99

✅ GT 기본 검증 통과!


“For each order we compute the F-1 score between the set of products you submit and the true set of products actually reordered. Your final score is the mean of these F-1 scores across all test-set orders.”

### -> 따라서 전체 주문 집합만을 가지고도 F1-score 계산 가능!

### Grade

이건 제가 바빠서... 테스트는 못 해봤습니다. 위에 gt_df와 교집합을 비교해서 f1score를 산출하는 방식이에요.

In [ ]:
# ------------------------------------------------------------------
#  제출 CSV (member_id, products) ↔ gt_df(member_id, products) 비교
#  Instacart Kaggle 방식 : 회원별 F-1 → 전체 평균
#  ※ gt_df 는 이미 Neo4j 쿼리로 얻어 두었고
#     columns = [member_id, last_order_id, last_order_number, products]
# ------------------------------------------------------------------

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

# ──────────────────────────────────────────
# 1. 제출 CSV 로드 : member_id, products
#    products = "pid1 pid2 …" 또는 "None"
# ──────────────────────────────────────────
csv_path = "submission.csv"

sub = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
sub.columns = sub.columns.str.strip().str.lower()      # 열 이름 정리

if {"member_id", "products"} - set(sub.columns):
    raise ValueError("CSV must 포함 member_id, products columns")

sub["member_id"] = sub["member_id"].astype(str).str.strip()  # 문자열화

def split_prod(cell: str) -> list[str]:
    cell = (cell or "").strip()
    return [] if cell.lower() == "none" or cell == "" else cell.split()

sub["products_pred"] = sub["products"].apply(split_prod)
sub = sub[["member_id", "products_pred"]]

print(f"제출 회원 수      : {len(sub):,}")

# ──────────────────────────────────────────
# 2. GT DataFrame 준비
#    (member_id, products_gt) 형태로 축소
# ──────────────────────────────────────────
gt = gt_df[["member_id", "products"]].copy()
gt.rename(columns={"products": "products_gt"}, inplace=True)

print(f"GT 회원 수        : {len(gt):,}")

# ──────────────────────────────────────────
# 3. member_id 기준 병합
# ──────────────────────────────────────────
merged = (
    pd.merge(gt, sub, on="member_id", how="inner")   # inner → 공통 회원만 평가
      .sort_values("member_id")
      .reset_index(drop=True)
)

print(f"공통 member_id 수 : {len(merged):,}")

if merged.empty:
    raise ValueError("GT와 제출 파일의 member_id 가 하나도 매칭되지 않았습니다.")

# ──────────────────────────────────────────
# 4. 전체 상품 vocabulary 생성
# ──────────────────────────────────────────
vocab = sorted(
    {pid for lst in merged["products_gt"]   for pid in lst} |
    {pid for lst in merged["products_pred"] for pid in lst}
)
print(f"상품 vocabulary 크기 : {len(vocab):,}")

# ──────────────────────────────────────────
# 5. MultiLabelBinarizer → sparse indicator
# ──────────────────────────────────────────
mlb = MultiLabelBinarizer(classes=vocab)

y_true = mlb.fit_transform(merged["products_gt"])     # CSR sparse
y_pred = mlb.transform(   merged["products_pred"])

# ──────────────────────────────────────────
# 6. 회원별 F-1 평균 (average="samples")
# ──────────────────────────────────────────
kaggle_f1 = f1_score(
    y_true,
    y_pred,
    average="samples",     # 각 회원 F1 → 평균
    zero_division=0        # GT=Pred=∅ → F1=1, 나머지 분모0 → 0
)

print(f"\n🎯 Kaggle-style macro F1-score : {kaggle_f1:.6f}")
